## Load in the data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd
from sklearn.svm import SVC
import cloudpickle
from tqdm import trange
from sklearn.metrics import roc_auc_score

import mat73

In [ ]:
my_dict = mat73.loadmat('../SingleRegion_Aggression_data.mat')

In [ ]:
TrainsetMouse = my_dict['TrainsetMouse']
mbeh_all2 = my_dict['mbeh_all2']
mcond_all2 = my_dict['mcond_all2']
mouse_all2 = my_dict['mouse_all2']
mpow_all2 = my_dict['mpow_all2']
mpow_all3s = my_dict['mpow_all3s']
mtimecondbeh2 = my_dict['mtimecondbeh2']
mu_all3 = my_dict['mu_all3']
testsetMouse = my_dict['testsetMouse']


In [ ]:
N_samples = len(mouse_all2)

mice_all = []
for mouse in TrainsetMouse:
    mice_all.append(mouse[0])
for mouse in testsetMouse:
    mice_all.append(mouse[0])
mice_all = np.array(mice_all)

trainset = []
for i in range(len(TrainsetMouse)):
    trainset.append(TrainsetMouse[i][0])

train_idxs = np.zeros(N_samples)
mouse_idxs = np.zeros(N_samples)
for i in range(N_samples):
    if mouse_all2[i][0] == 'Mouse048':
        train_idxs[i] = -1
        continue
    if mouse_all2[i][0] in trainset:
        train_idxs[i] = 1
    mouse_idxs[i] = np.where(mice_all==mouse_all2[i][0])[0][0]

In [ ]:
testsetMouse

## Create the function for analyzing data

In [ ]:
def analyze_data(idxs_pos,idx_neg,baseSaveName):
    selection_indices = idxs_pos|idx_neg
    y = np.zeros(N_samples)
    y[idxs_pos] = 1
    mpower_reduced = mpow_all2[:,:,selection_indices]
    y_reduced = y[selection_indices]
    train_idxs_reduced = train_idxs[selection_indices]
    y_train = y_reduced[train_idxs_reduced==1]
    y_test = y_reduced[train_idxs_reduced==0]
    m_idx_unique_test = np.unique(mouse_idxs[train_idxs==0])
    mouse_idxs_reduced = mouse_idxs[selection_indices]
    m_test = mouse_idxs_reduced[train_idxs_reduced==0]
    model_list = []
    
    test_aucs_mouse = np.zeros((11,9))
    for i in trange(11):
        XT = np.squeeze(mpower_reduced[:,i,:])
        X = np.transpose(XT)
        X = X*10
        X[X>6] = 6
        Xtrain = X[train_idxs_reduced==1]
        Xtest = X[train_idxs_reduced==0]
        model = SVC(cache_size=1000)
        model.fit(Xtrain,y_train)
        Y_hat = model.decision_function(Xtest)
        model_list.append(model)
        for j in range(9):
            print(mice_all[20+j])
            test_aucs_mouse[i,j] = roc_auc_score(y_test[m_test==20+j],
                                        np.squeeze(Y_hat[m_test==20+j]))
    myDict = {'models':model_list}
    mname = baseSaveName + '_Models.p'
    #with open(mname,'wb') as f:
    #    cloudpickle.dump(myDict,f)
    #cname = baseSaveName + '_ROCs.csv'
    #np.savetxt(cname,test_aucs_mouse,fmt='%0.8f',delimiter=',')

## Isolation experiment

In [ ]:
idxs_pos = (mtimecondbeh2[:,0]<240)&(mtimecondbeh2[:,1]==0)
idx_neg = ((mcond_all2==4)&(mbeh_all2>0))|((mcond_all2==6)&(mbeh_all2==2))|((mcond_all2==8)&(mbeh_all2==2))

analyze_data(idxs_pos,idx_neg,'Isolation')

## Male Non Agg vs FemaleCast

In [ ]:
idxs_pos = (mcond_all2==4)&(mbeh_all2==2)
idx_neg = ((mcond_all2==4)&(mbeh_all2==1))|((mcond_all2==6)&(mbeh_all2==2))|((mcond_all2==8)&(mbeh_all2==2))
analyze_data(idxs_pos,idx_neg,'MaleNonAggVsFemaleCast')

## AggressionVsProsocial

In [ ]:
idxs_pos = (mcond_all2==4)&(mbeh_all2==1)
idx_neg = ((mcond_all2==4)|(mcond_all2==6)|(mcond_all2==8))&(mbeh_all2==2)
analyze_data(idxs_pos,idx_neg,'AgressionVsProsocial')

## CastratedVsMaleFem

In [ ]:
idxs_pos = (mcond_all2==8)&(mbeh_all2==2)
idx_neg = ((mcond_all2==4)&(mbeh_all2>0))|((mcond_all2==6)&(mbeh_all2==2))
analyze_data(idxs_pos,idx_neg,'CastratedVsMaleFem')

## FemaleVsMaleCastrated

In [ ]:
idxs_pos = (mcond_all2==6)&(mbeh_all2==2)
idx_neg = ((mcond_all2==4)&(mbeh_all2>0))|((mcond_all2==8)&(mbeh_all2==2))
analyze_data(idxs_pos,idx_neg,'FemaleVsMaleCastrated')